In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors
import scrublet as scr
import session_info

In [2]:
import session_info
session_info.show()

/home/Data_Drive_8TB/ahwodn/.conda/envs/ahwodn_2/lib/python3.10/site-packages/session_info/main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  mod_version = _find_version(mod.__version__)
/home/Data_Drive_8TB/ahwodn/.conda/envs/ahwodn_2/lib/python3.10/site-packages/session_info/main.py:213: UserWarning: The '__version__' attribute is deprecated and will be removed in MarkupSafe 3.1. Use feature detection, or `importlib.metadata.version("markupsafe")`, instead.
  mod_version = _find_version(mod.__version__)


In [3]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.settings.set_figure_params(dpi=300, facecolor='white', color_map = 'RdPu',)

# 0. Data load

In [4]:
cap=sc.read_h5ad('/path/to/h5ad_annotated/_sc_Myeloid.h5ad')

In [5]:
cap

AnnData object with n_obs × n_vars = 297446 × 4000
    obs: 'Data', 'Patient', 'Sample', 'Organ_orig', 'Tissue_site', 'Subtype', 'Type', 'Platform', 'Kit_version', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_counts', 'n_genes', 'batch_key', 'sub_anno'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection', 'mean', 'std'
    uns: 'hvg', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'res_1.4', 'res_1.4_colors', 'sub_anno_colors', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap', 'X_umap_uncorrected'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [6]:
unique_values = list(cap.obs['sub_anno'].unique())
print("Total unique values:", len(unique_values))
print("Unique values:")
for value in unique_values:
    print(value)

Total unique values: 15
Unique values:
CXCL3+ Macro
CD14+ Mono
CD1C+ cDC
FOLR2+ Macro
HSPA6+ Macro
SPP1+ cycMacro
CD16+ Mono
Macro
LILRA4+ pDC
CLEC9A+ cDC
TNFSF10+ Macro
SPP1+ CXCL3+ Macro
SPP1+ Macro
SPP1+ MT1+ Macro
LAMP3+ cDC


# 1-2. ORA

In [7]:
# sub_anno의 고유 그룹 리스트 가져오기
sub_anno_groups = cap.obs['sub_anno'].unique()

# 각 그룹별 DEG 리스트 생성
gene_lists = {
    group: list(sc.get.rank_genes_groups_df(cap, group=group, 
                                            pval_cutoff=0.01, log2fc_min=1.5)['names'])
    for group in sub_anno_groups
}

In [8]:
# gene_lists에서 값이 비어있는 key 찾기
empty_keys = [key for key, value in gene_lists.items() if not value]

# 결과 출력
empty_keys

[]

# ORA

In [9]:
#GSEA
import gseapy as gp

In [10]:
from gseapy import enrichr

# 결과를 저장할 데이터프레임 초기화
final_results_df = pd.DataFrame()

# GSEA를 수행할 gene set 목록
gene_sets_list = [
    'MSigDB_Hallmark_2020',
]

# 각 gene set에 대해 반복적으로 GSEA 수행
for gene_set in gene_sets_list:
    for organ, gene_list in gene_lists.items():
        enr_res = enrichr(gene_list=gene_list,
                          gene_sets=[gene_set],
                          background=None,
                          outdir=None,
                          verbose=True)

        # 결과에 organ 및 gene set 정보를 추가
        enr_res.results['cluster_name'] = f'{organ}'
        enr_res.results['Gene_set'] = gene_set

        # 결과를 병합
        final_results_df = pd.concat([final_results_df, enr_res.results], ignore_index=True)


2026-09-09 20:54:14,445 [INFO] Run: MSigDB_Hallmark_2020 
2026-09-09 20:54:16,438 [INFO] Done.
2026-09-09 20:54:17,789 [INFO] Run: MSigDB_Hallmark_2020 
2026-09-09 20:54:19,866 [INFO] Done.
2026-09-09 20:54:21,126 [INFO] Run: MSigDB_Hallmark_2020 
2026-09-09 20:54:23,324 [INFO] Done.
2026-09-09 20:54:24,578 [INFO] Run: MSigDB_Hallmark_2020 
2026-09-09 20:54:26,521 [INFO] Done.
2026-09-09 20:54:27,873 [INFO] Run: MSigDB_Hallmark_2020 
2026-09-09 20:54:29,779 [INFO] Done.
2026-09-09 20:54:31,034 [INFO] Run: MSigDB_Hallmark_2020 
2026-09-09 20:54:33,750 [INFO] Done.
2026-09-09 20:54:34,998 [INFO] Run: MSigDB_Hallmark_2020 
2026-09-09 20:54:37,150 [INFO] Done.
2026-09-09 20:54:38,415 [INFO] Run: MSigDB_Hallmark_2020 
2026-09-09 20:54:40,289 [INFO] Done.
2026-09-09 20:54:41,533 [INFO] Run: MSigDB_Hallmark_2020 
2026-09-09 20:54:44,081 [INFO] Done.
2026-09-09 20:54:45,384 [INFO] Run: MSigDB_Hallmark_2020 
2026-09-09 20:54:47,664 [INFO] Done.
2026-09-09 20:54:48,921 [INFO] Run: MSigDB_Hallmar

In [11]:
from plotting import (apply_style, pval_to_stars, row_zscore,
                          plot_enrichment_clustermap_combined_score_dif)

apply_style()

In [12]:
dd_sub=cap.copy()

In [13]:

sns.set_style("white")

plt.rcParams["font.family"] = "Arial"
plt.rcParams["mathtext.fontset"] = "custom"   # 수식($...$)도 Arial로
plt.rcParams["mathtext.rm"] = "Arial"
plt.rcParams["mathtext.it"] = "Arial:italic"
plt.rcParams["mathtext.bf"] = "Arial:bold"
plt.rcParams["pdf.fonttype"] = 42             # PDF에 텍스트를 편집 가능한 형태로 임베드 (저널 제출용)

annot_ordering = [
    "Macro",
    "FOLR2+ Macro",
    "CXCL3+ Macro",
    "SPP1+ Macro",
    "SPP1+ CXCL3+ Macro",
    "SPP1+ cycMacro",
    "SPP1+ MT1+ Macro",
    "TNFSF10+ Macro",
    "HSPA6+ Macro",
    "CD14+ Mono",
    "CD16+ Mono",
    "CD1C+ cDC",
    "LAMP3+ cDC",
    "CLEC9A+ cDC",
    "LILRA4+ pDC",
]



base_color_map = dict(
    zip(dd_sub.obs["sub_anno"].cat.categories, dd_sub.uns["sub_anno_colors"])
)

cluster_color_map = {k: base_color_map[k] for k in annot_ordering}

final_results_df['-log10(Adjusted P-value)'] = -np.log10(final_results_df['Adjusted P-value'])

top_terms_df = final_results_df.groupby('cluster_name').apply(
    lambda x: x.nlargest(5, '-log10(Adjusted P-value)')
).reset_index(drop=True)

final_results_df_dd2 = top_terms_df.dropna(subset=["cluster_name"])

g, plot_df, row_meta, raw_score_df = plot_enrichment_clustermap_combined_score_dif(
    final_results_df_dd=final_results_df_dd2,
    gene_sets_to_use=["MSigDB_Hallmark_2020"],
    cluster_order=annot_ordering,
    top_k=3,
    score_col='-log10(Adjusted P-value)',
    cbar_label=r'-log$_{10}$(Adjusted $P$-value)',
    title='MSigDB Hallmark 2020',
    use_row_zscore=False,
    cluster_color_map=cluster_color_map,
    col_colors_gap=0.002,
    col_colors_ratio=0.03,
    figsize=(34, 35),
    vmax=5,
    cmap='RdPu_r',
    dpi=700,
    #save_path="./figures/hallmark_ora_specificity_top3.pdf",
)


/tmp/ipykernel_1359764/350516693.py:38: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  top_terms_df = final_results_df.groupby('cluster_name').apply(
/home/Data_Drive_8TB/ahwodn/Jupyter/cllcraft/Jupyter/SMC_korean_pancancer_jupyter/alldata_concat/Hot_cold_2024/10_lastest_Hot_cold_2024_backup/15_코드정리/sc_수정_ver2/plotting.py:205: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  df.pivot_table(
/home/Data_Drive_8TB/ahwodn/Jupyter/cllcraft/Jupyter/SMC_korean_pancancer_jupyter/alldata_concat/Hot_cold_2024/10_lastest_Hot_cold_2024_backup/15_코드정리/sc_수정

In [14]:
top_terms_df.to_csv('Myeloid_ora_result.csv')